In [1]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer
from rdkit.Chem.SaltRemover import SaltRemover

from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import Chem
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda:0


In [ ]:
def find_model_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates = [
            base / "TKAM_heptox_models",
            base / "code" / "predict" / "TKAM_heptox_models",
            base / "code" / "predict" / "predict_models" / "TKAM_heptox_models",
        ]
        for candidate in candidates:
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Cannot find TKAM_heptox_models")

MODEL_ROOT = find_model_root()
MODEL_DIRS = sorted(
    [p for p in MODEL_ROOT.iterdir() if p.is_dir() and (p / "pytorch_model.bin").exists()],
    key=lambda p: p.name,
)

if len(MODEL_DIRS) != 5:
    raise RuntimeError(f"Expected 5 models in {MODEL_ROOT}, found {len(MODEL_DIRS)}")

print(f"Model root: {MODEL_ROOT}")
for model_dir in MODEL_DIRS:
    print(f"- {model_dir.name}")

Model root: C:\Users\29715\Desktop\github代码\code\predict\TKAM_heptox_models
- s4_curriculum_stage2_seed_123_replay_0.8_seed123
- s4_curriculum_stage2_seed_2026_replay_0.8_seed2026
- s4_curriculum_stage2_seed_42_replay_0.8_seed42
- s4_curriculum_stage2_seed_888_replay_0.8_seed888
- s4_curriculum_stage2_seed_999_replay_0.8_seed999


In [9]:
class MultiTaskChemBERTa(nn.Module):
    def __init__(self, model_dir, task_names, head_dropout=0.5, head_hidden_dim=512):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_dir)
        self.encoder = AutoModel.from_config(self.config)
        self.task_names = list(task_names)
        self.heads = nn.ModuleDict()
        self.task_log_vars = nn.ParameterDict()

        for task_name in self.task_names:
            self.heads[task_name] = nn.Sequential(
                nn.Dropout(head_dropout),
                nn.Linear(self.config.hidden_size, head_hidden_dim),
                nn.Tanh(),
                nn.Dropout(head_dropout),
                nn.Linear(head_hidden_dim, 1),
            )
            self.task_log_vars[task_name] = nn.Parameter(torch.zeros(()))

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        return {
            task_name: self.heads[task_name](pooled_output).squeeze(-1)
            for task_name in self.task_names
        }


def load_model(model_dir, device):
    config = AutoConfig.from_pretrained(model_dir)
    task_names = list(getattr(config, "task_names", []))
    if not task_names:
        raise ValueError(f"No task_names found in {model_dir / 'config.json'}")

    model = MultiTaskChemBERTa(
        model_dir=model_dir,
        task_names=task_names,
        head_dropout=getattr(config, "head_dropout", 0.5),
        head_hidden_dim=getattr(config, "head_hidden_dim", 512),
    )
    state_dict = torch.load(
        model_dir / "pytorch_model.bin",
        map_location=device,
        weights_only=True,
    )
    model.load_state_dict(state_dict, strict=False)
    model.to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    return model, tokenizer, config

In [10]:
def predict_five_model_ensemble(smiles, task_name="Invivo_PFAS", max_length=128):
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("SMILES must be a non-empty string")
    invivo_rows = []
    bioassay_rows = []

    for model_dir in MODEL_DIRS:
        model, tokenizer, config = load_model(model_dir, DEVICE)
        if task_name not in model.task_names:
            raise KeyError(f"{task_name} is not available in {model_dir.name}")

        encoded = tokenizer(
            [smiles.strip()],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

        with torch.inference_mode():
            logits_by_task = model(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
            )
            invivo_probability = torch.sigmoid(
                logits_by_task[task_name]
            )[0].item()

        invivo_rows.append({
            "model": model_dir.name,
            "task": task_name,
            "probability": invivo_probability,
            "model_threshold": getattr(config, "mcc_best_thresh", None),
        })

        pruned_tasks = set(getattr(config, "pruned_tasks", []))
        active_bioassay_tasks = [
            task for task in model.task_names
            if task.startswith("AID:") and task not in pruned_tasks
        ]
        for bioassay_task in active_bioassay_tasks:
            bioassay_rows.append({
                "task": bioassay_task,
                "model": model_dir.name,
                "probability": torch.sigmoid(
                    logits_by_task[bioassay_task]
                )[0].item(),
            })

        del model, tokenizer, config, encoded, logits_by_task
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    invivo_details = pd.DataFrame(invivo_rows)
    invivo_mean = float(invivo_details["probability"].mean())

    bioassay_long = pd.DataFrame(bioassay_rows)
    bioassay_summary = (
        bioassay_long.groupby("task", as_index=False)["probability"]
        .mean()
        .rename(columns={"probability": "mean_probability"})
        .sort_values("mean_probability", ascending=False)
        .reset_index(drop=True)
    )
    return {
        "invivo_mean": invivo_mean,
        "invivo_details": invivo_details,
        "bioassay_predictions": bioassay_summary,
    }


def predict_five_model_mean(smiles, task_name="Invivo_PFAS", max_length=128):
    return predict_five_model_ensemble(
        smiles, task_name, max_length
    )["invivo_mean"]

In [ ]:
SALT_REMOVER = SaltRemover()
UNCHARGER = rdMolStandardize.Uncharger()
CHOOSER = rdMolStandardize.LargestFragmentChooser()

def standardize_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return np.nan
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.nan
        mol = SALT_REMOVER.StripMol(mol, dontRemoveEverything=True)
        if "." in smiles:
            mol = CHOOSER.choose(mol)    
        if "+" in smiles or "-" in smiles:      
            mol = UNCHARGER.uncharge(mol) 
        return Chem.MolToSmiles(mol, isomericSmiles=True,canonical=True)
        
    except Exception:
        return np.nan

In [ ]:
SMILES = "CN(C)CCCNS(=O)(=O)CCC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F"
TASK_NAME = "Invivo_PFAS"
SMILES_STANDARDIZED = standardize_smiles(SMILES)
prediction = predict_five_model_ensemble(SMILES_STANDARDIZED, TASK_NAME)
print(f"{TASK_NAME}_pred: {prediction['invivo_mean']:.6f}")
display(prediction["bioassay_predictions"])

Invivo_PFAS_pred: 0.510707


,task,mean_probability
0,AID:720637,0.889844
1,AID:1032,0.884817
2,AID:1346985,0.692043
3,AID:743219,0.642352
4,AID:504444,0.625000
5,AID:588692,0.397336
6,AID:1347034,0.258395
7,AID:743416,0.230450
8,AID:504648,0.089296
9,AID:489027,0.078666
